# 6.3.6 Agent Frameworks

Build a minimal LangChain-style chain: PromptTemplate → LLM stub → OutputParser. Demonstrate chain composition and pipeline patterns.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)

class PromptTemplate:
    def __init__(self, tmpl): self.tmpl = tmpl
    def format(self, **kw):
        r = self.tmpl
        for k, v in kw.items(): r = r.replace(f'{{{k}}}', str(v))
        return r

class LLMStub:
    def __init__(self): self.log = []
    def call(self, prompt):
        resp = 'positive' if 'classify' in prompt.lower() else 'entity1, entity2'
        self.log.append({'in': len(prompt), 'out': len(resp)})
        return resp

class OutputParser:
    def __init__(self, mode='strip'): self.mode = mode
    def parse(self, t):
        if self.mode == 'list': return [x.strip() for x in t.split(',')]
        if self.mode == 'bool': return 'positive' in t.lower()
        return t.strip()

llm = LLMStub()
pt = PromptTemplate('Classify: "{text}". classify the sentiment.')
op = OutputParser('bool')

for text in ['I love this!', 'This is terrible.']:
    prompt = pt.format(text=text)
    response = llm.call(prompt)
    result = op.parse(response)
    print(f'  "{text}" → {result}')

In [ ]:
steps = ['PromptTemplate', 'LLM Call', 'OutputParser']
latencies = [2, 850, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(steps, latencies, color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0].set_yscale('log')
axes[0].set(ylabel='Latency (ms, log)', title='Chain Step Latency')
axes[0].grid(True, axis='y', alpha=0.3)

if llm.log:
    x = np.arange(len(llm.log))
    ins = [c['in'] for c in llm.log]; outs = [c['out'] for c in llm.log]
    axes[1].bar(x-0.2, ins, 0.4, label='Prompt', color='steelblue')
    axes[1].bar(x+0.2, outs, 0.4, label='Response', color='darkorange')
    axes[1].set(xlabel='Call #', ylabel='Length', title='LLM I/O Lengths'); axes[1].legend()
plt.tight_layout()
plt.savefig('output/agent_frameworks.png')
print('Saved agent_frameworks.png')